# 02 — REDSEA spillover correction and selected expression

This notebook inspects the production `redsea` and `expression` artifacts. REDSEA owns qptiff rasterization, contact correction, compartment output, alignment checks, and fail-fast donor execution. The marker registry then selects exactly one authoritative intensity per marker. No compensation or source-selection math is reimplemented here.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from phenocycler.artifacts import StageManifest
from phenocycler.config import load_config
from phenocycler.expression import read_single_partition
from phenocycler.pipeline import RunContext, resolve_run_context, run_stage, status

CONFIG_PATH = None
cfg = load_config(CONFIG_PATH)
context: RunContext = resolve_run_context(cfg)
print(f"run_id={context.run_id}  donors={len(context.donors)}  root={context.run_root}")

In [ ]:
status_code = status(context)
print(f"status return code: {status_code}")

## Optional production execution

`redsea` requires current ingest and geometry manifests. `expression` requires current ingest, geometry, REDSEA, and marker-registry fingerprints. Existing valid stages are checked rather than recomputed.

In [ ]:
RUN_STAGES = False

if RUN_STAGES:
    for stage_name in ("redsea", "expression"):
        run_stage(context, stage_name)
else:
    print("Inspection only. Set RUN_STAGES=True to run production REDSEA and expression selection.")

In [ ]:
manifest_rows = []
for stage_name in ("redsea", "expression"):
    path = context.stage_manifest_path(stage_name)
    if path.exists():
        manifest = StageManifest.read_json(path)
        manifest_rows.append({
            "stage": stage_name,
            "method_version": manifest.method_version,
            "donors": len(manifest.completed_donors),
            "rows": manifest.output.total_rows,
            "columns": len(manifest.output.schema),
            "schema": manifest.output.schema_sha256[:12],
            "objects": manifest.output.object_id_sha256[:12],
            "content": manifest.content_id[:12],
        })
display(pd.DataFrame(manifest_rows))

## Authoritative marker source

The registry—not notebook column guessing—declares each marker's compartment and whether its value is REDSEA-corrected, REDSEA passthrough, or uncompensated. Markers absent from a donor's acquisition panel remain null with an availability reason.

In [ ]:
registry_rows = [
    {
        "marker": marker.name,
        "kind": marker.kind,
        "compartment": marker.compartment,
        "spillover_policy": marker.spillover_policy,
        "measurement_column": marker.measurement_column,
        "reference": marker.reference,
        "calibration_status": marker.calibration_status,
    }
    for marker in context.registry.markers
]
display(pd.DataFrame(registry_rows))

## Inspect REDSEA and selected expression

The REDSEA table is compartment-resolved. The selected-expression table is intentionally narrow: one value per registered marker plus cell identity, spatial context, and geometry eligibility. A corrected value may increase or decrease because the configured REDSEA operator includes both boundary reinforcement and neighbor-spillover subtraction.

In [ ]:
DONOR = context.donors[0]
required = (context.stage_manifest_path("redsea"), context.stage_manifest_path("expression"))
if all(path.exists() for path in required):
    redsea = read_single_partition(context.config.redsea_dir, DONOR)
    expression = read_single_partition(context.config.selected_expression_dir, DONOR)
    print(f"donor {DONOR}: REDSEA={redsea.shape}, selected expression={expression.shape}")
    redsea_measurements = [column for column in redsea if "__" in column]
    display(redsea.loc[:, ["object_id", *redsea_measurements[:10]]].head())
    context_columns = [
        column for column in (
            "donor_id", "object_id", "image", "cell_region",
            "qc_analysis_eligible", "qc_estimation_eligible"
        ) if column in expression
    ]
    marker_columns = [marker.name for marker in context.registry.markers if marker.name in expression]
    display(expression.loc[:, [*context_columns, *marker_columns[:8]]].head())
else:
    print("REDSEA/expression artifacts are not complete yet.")

In [ ]:
availability_path = context.config.audit_dir / "expression_availability.parquet"
if availability_path.exists():
    availability = pd.read_parquet(availability_path)
    display(
        availability.groupby(["available", "reason"], dropna=False).size()
        .rename("donor_markers").to_frame()
    )
    display(availability.loc[availability["donor_id"].astype(str).eq(DONOR)].head(20))
else:
    print("Expression availability audit is not present.")

## Handoff

Proceed to calibration only when `redsea` and `expression` are `CURRENT` and the availability audit agrees with the acquisition panels. The selected-expression artifact is the sole intensity input to reference-control selection and donor-marker calibration.